In [ ]:
# from google.colab import drive
# drive.mount('/content/drive/')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import math
from scipy import stats

from sklearn.preprocessing import OrdinalEncoder
import missingno as msno
import statsmodels.stats as sms
from sklearn import feature_selection
from sklearn import preprocessing
from sklearn import pipeline
from sklearn import compose
from sklearn import impute
from sklearn import model_selection
from sklearn import decomposition
import wandb
import wave
import os
import time

In [ ]:
data = pd.read_csv("speeddating.csv", na_values='?')

In [ ]:
data.info()

In [ ]:
data.columns.tolist()
data['has_null'].groupby(data['has_null']).agg("count")

In [ ]:
data.head()

In [ ]:
data.describe(include='all')

In [ ]:
duplicates = data.duplicated()
data[duplicates].shape[0]

In [ ]:
data.isnull().sum().sum()

In [ ]:
# data.dropna(inplace=True)

In [ ]:
data.head()

In [ ]:
def remove_outliers(df: pd.DataFrame):
    df_cleaned = df.copy()
    changed_columns = []
    numeric_cols = df.select_dtypes(include=['number']).columns
    for column_name in numeric_cols:
        Q1 = df[column_name].quantile(0.25)
        Q3 = df[column_name].quantile(0.75)
        IQR = Q3 - Q1
        mask = (df[column_name] >= Q1 - 1.5 * IQR) & (df[column_name] <= Q3 + 1.5 * IQR)
        df_cleaned[column_name] = df[column_name][mask]
        if mask.sum() < len(df[column_name]):
            changed_columns.append(column_name)
    return df_cleaned, changed_columns

In [ ]:
df_cleaned, changed_columns = remove_outliers(data)

In [ ]:
data.columns.tolist()
data['has_null'].groupby(data['has_null']).agg("count")

In [ ]:
changed_columns

No outliers

In [ ]:
sns.histplot(x=data['age'], bins=15, label='initial')
sns.histplot(x=df_cleaned['age'], bins=15, label='no outliers')
plt.legend()

In [ ]:
sns.boxplot(x=data['age'], y=data['gender'])


In [ ]:
sns.boxplot(x=df_cleaned['age'], y=df_cleaned['gender'])

In [ ]:
sns.scatterplot(data=data, x='age', y='attractive', alpha=0.5)
plt.title('Age vs attr of partner')
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(data=data, x='gender', y='age', hue='met')
plt.title('Age vs met by gender')
plt.tight_layout()
plt.show()

In [ ]:
race_counts = data.groupby(['race', 'race_o']).size().unstack(fill_value=0)

plt.figure(figsize=(10, 7))
sns.heatmap(race_counts, annot=True, fmt='d', cmap='Blues')
plt.title('Frequency of Race Pairings')
plt.xlabel('Partner\'s Race')
plt.ylabel('Your Race')
plt.tight_layout()
plt.show()

In [ ]:
plot_df = data[
    ["gender"] + [
        'sports',
 'tvsports',
 'exercise',
 'dining',
 'museums',
 'art',
 'hiking',
 'gaming',
 'clubbing',
 'reading',
 'tv',
 'theater',
 'movies',
 'concerts',
 'music',
 'shopping',
 'yoga'
 ]
    ].melt(id_vars="gender", var_name="interest", value_name="score")
plt.figure()
sns.barplot(data=plot_df, x="interest", y="score", hue="gender", estimator=np.mean)
plt.xticks(rotation=45, ha="right")
plt.title("Top interests by average usage, male vs female")
plt.tight_layout()
plt.show()

met and match

In [ ]:
g1= data.groupby("decision")[["met","match"]].mean().reset_index()
g1

In [ ]:
g2 = data.groupby("decision_o")[["met","match"]].mean().reset_index()
g2

In [ ]:
g3 = data.groupby(["decision","decision_o"])[["met","match"]].mean().reset_index()
g3

In [ ]:
na_data = data.isna().sum()
na_percent = (na_data / len(data)) * 100
na_percent[na_percent > 95]

## Analyse Missing Data

In [ ]:
plt.figure(figsize=(12, 6))
msno.matrix(data)
plt.title('Missing Values Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(data.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

In [ ]:
missing_values = data.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values.sort_values(inplace=True)

plt.figure(figsize=(14, 10))
missing_values.plot(kind='barh', color='orange')
plt.title('Missing Values Count by Column', fontsize=12)
plt.xlabel('Number of Missing Values', fontsize=10)
plt.ylabel('Columns', fontsize=10)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

there is a big number of missing values in the column expected_num_interests_in_me. Additionally, we did not find any relation to this missigness, so we assume to drop it

In [ ]:
data.drop(columns=['expected_num_interested_in_me'], inplace=True)

returning to the patterns of missigness, it is seem, that there is MAR missigness. We see, that some data can be missed by the 80% of the string, but some occured just randomly. We consider to use imputation MICE algorithm for dealing with missing data

In [ ]:
def get_missing_columns(df, columns_type):
  columns = df.select_dtypes(include=columns_type)
  has_missing_mask = columns.isnull().any() 
  missing_columns = has_missing_mask[has_missing_mask].index.tolist()
  return missing_columns

In [ ]:
numerical_missing_columns = get_missing_columns(data, ['number'])
categorical_missing_values = get_missing_columns(data, ['object', 'string'])

print(f"Numerical missing values: {numerical_missing_columns}")
print(f"Categorical missing values: {categorical_missing_values}")

## Analyse categorical data

In [ ]:
categorical_columns = data.select_dtypes(include=['object', 'category']).columns
categorical_columns

### Columns cleaning

We found out, that there are the main parameters, that are observed in the dataset which are:\
intelligence, attractive, sincere, funny, ambition, shared_interests

I noticed the errors in some namings. So in this section we will transform the namings to fit the names and for the easier namings

In [ ]:
parameters = ["intelligence", "attractive", "sincere", "funny", "ambition", "shared_interests"]

In [ ]:
unified_parameters = {}
for param in parameters:
  param_correlations = []
  partner_param_importance = f"pref_o_{param}"
  my_param_importance = f"{param}_important"
  partner_param_evaluation = f"{param}_o"
  my_param_evaluation = f"{param}_partner"
  params = [partner_param_importance, my_param_importance, partner_param_evaluation, my_param_evaluation]
  unified_parameters[param] = params
  
  for p in params:
      if p not in data.columns:
          print(f"Column {p} not found in data.")

In [ ]:
unified_d_parameters = {}
for param in parameters:
  param_correlations = []
  partner_param_importance = f"d_pref_o_{param}"
  my_param_importance = f"d_{param}_important"
  partner_param_evaluation = f"d_{param}_o"
  my_param_evaluation = f"d_{param}_partner"
  params = [partner_param_importance, my_param_importance, partner_param_evaluation, my_param_evaluation]
  unified_d_parameters[param] = params
  
  for p in params:
      if p not in data.columns:
          print(f"Column {p} not found in data.")

In [ ]:
cleaned_df = data.copy()
cleaned_df = cleaned_df.rename(columns={
  "intellicence_important": "intelligence_important",
  "d_intellicence_important": "d_intelligence_important",
  "sinsere_o": "sincere_o",
  "d_sinsere_o": "d_sincere_o",
  "pref_o_ambitious": "pref_o_ambition",
  "d_pref_o_ambitious": "d_pref_o_ambition",
  "ambtition_important": "ambition_important",
  "d_ambtition_important": "d_ambition_important",
  "ambitous_o": "ambition_o",
  "d_ambitous_o": "d_ambition_o",
})

cleaned_df.head()

We cleaned the columns naming, now we will reame them to be easier for understanding

In [ ]:
def rename_to_prefered_name(df, column_name, prefered_name):
    df.rename(columns={column_name: prefered_name}, inplace=True)

In [ ]:
unified_parameters = {}
for param in parameters:
  rename_to_prefered_name(cleaned_df, f"pref_o_{param}", f"{param}_important_o")
  rename_to_prefered_name(cleaned_df, f"{param}_o", f"p_eval_of_my_{param}")
  rename_to_prefered_name(cleaned_df, f"{param}_partner", f"my_eval_of_p_{param}")
  partner_param_importance = f"{param}_important_o"
  my_param_importance = f"{param}_important"
  partner_param_evaluation = f"p_eval_of_my_{param}"
  my_param_evaluation = f"my_eval_of_p_{param}"
  params = [partner_param_importance, my_param_importance, partner_param_evaluation, my_param_evaluation]
  unified_parameters[param] = params

unified_parameters

In [ ]:
unified_d_parameters = {}

for param in parameters:
  rename_to_prefered_name(cleaned_df, f"d_pref_o_{param}", f"d_{param}_important_o")
  rename_to_prefered_name(cleaned_df, f"d_{param}_o", f"d_p_eval_of_my_{param}")
  rename_to_prefered_name(cleaned_df, f"d_{param}_partner", f"d_my_eval_of_p_{param}")
  partner_param_importance = f"d_{param}_important_o"
  my_param_importance = f"d_{param}_important"
  partner_param_evaluation = f"d_p_eval_of_my_{param}"
  my_param_evaluation = f"d_my_eval_of_p_{param}"
  params = [partner_param_importance, my_param_importance, partner_param_evaluation, my_param_evaluation]
  unified_d_parameters[param] = params

unified_d_parameters

In [ ]:
print(cleaned_df.columns.tolist())

In [ ]:
for param in parameters:
    cols = unified_parameters[param]
    for col in cols:
        missing_values_count = cleaned_df[cleaned_df[col].isna()].shape[0]
        percentage_missing = (missing_values_count / len(cleaned_df)) * 100
        print(f"{col}: {missing_values_count} missing values; {percentage_missing:.2f}%")

In [ ]:
threshold = 0.8
min_non_na = int((1 - threshold) * len(cleaned_df.columns))
cleaned_df = cleaned_df.dropna(thresh=min_non_na)

In [ ]:
for param in parameters:
    cols = unified_parameters[param]
    for col in cols:
        missing_values_count = cleaned_df[cleaned_df[col].isna()].shape[0]
        if missing_values_count == 0:
            print(f"{col}: No missing values")
            continue
        percentage_missing = (missing_values_count / len(cleaned_df)) * 100
        print(f"{col}: {missing_values_count} missing values; {percentage_missing:.2f}%")

### Check for the data content

In [ ]:
categorical_df = cleaned_df.select_dtypes(include=['object', 'category'])
categorical_df.head()

In [ ]:
categorical_columns = categorical_df.columns.tolist()
categorical_columns

In [ ]:
categorical_df.nunique()

In [ ]:
cols_unique_values = {}

for col in categorical_df.columns:
    unique_vals = categorical_df[col].unique()
    print(f"{col}: {len(unique_vals)}\n{unique_vals}\n")


According to the unique values represented for each categorical data field now we can divide them into 4 categorical group types:
1. Nominal data: gender, race, race_o, field (which is probably represent the job of person)
2. Ordinal data: - all the left fields are related to ratio data, since they cannot be less than 0 and represent some intervals. Despite, that these data re represented as ranges-string thay are follow in ascending order, so we can speak about them as ordinal data.

According to the received info we can proceed on working on data by the type of category.

### Nominal data

#### gender

In [ ]:
match_between_genders = cleaned_df.groupby(['match', 'gender']).size().unstack(fill_value=0)
match_between_genders

In [ ]:
match_percentages = (cleaned_df.groupby(['match', 'gender']).size().unstack(fill_value=0)
                     .div(cleaned_df.groupby('match').size(), axis=0) * 100)
match_percentages

In [ ]:
plt.figure(figsize=(10, 6))
ax = match_between_genders.plot(kind='bar', color=["#FF00E1", "#0800FF"])
plt.title('Distribution of Match by Gender')
plt.xlabel('Match')
plt.ylabel('Count')
plt.legend(title='Gender')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)

for container in ax.containers:
    ax.bar_label(container, label_type='edge')


plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
ax = match_percentages.plot(kind='bar', color=["#FF00E1", "#0800FF"])
plt.title('Distribution of Match by Gender')
plt.xlabel('Match')
plt.ylabel('Count')
plt.legend(title='Gender')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", label_type='edge')


plt.show()

In [ ]:
gender_encoded = pd.get_dummies(categorical_df, columns=['gender'])
gender_encoded.head()

It is clearly seen from the unique values, that we can indicate about 12 types of jobs. We will try to cluster them to unify to some 12 categories for the better processing them in the future model

### Correlations between variables and match

1. Use chi-square test to to determine if there is a significant association between each variable and target(match)
2. On the base of chi-square test result use Cramer's V test to measure the association between each variable and match - calculation of correlations 

In [ ]:
def chi_square_test(var1, var2):
    confusion_matrix = pd.crosstab(var1, var2)
    chi2, _, _, _ = stats.chi2_contingency(confusion_matrix)
    return chi2, confusion_matrix

In [ ]:
def cramerV(label, x):
    chi2, confusion_matrix = chi_square_test(label, x)
    n = confusion_matrix.sum().sum()
    minDim = min(confusion_matrix.shape) - 1
    V = np.sqrt((chi2 / n) / minDim)
    return V

In [ ]:
correlation_results = {}

for col in categorical_columns:
    correlation_results[col] = cramerV(cleaned_df['match'], categorical_df[col])

categorical_correlations = pd.DataFrame.from_dict(correlation_results, orient='index', columns=['correlation'])
print(categorical_correlations.sort_values(by='correlation', ascending=False))

In [ ]:
categorical_correlations_sorted = categorical_correlations.sort_values('correlation', ascending=False)
cmap = sns.diverging_palette(110, 10, as_cmap=True)

n_variables = len(categorical_correlations_sorted)
plt.figure(figsize=(8, max(6, n_variables * 0.4)))
sns.heatmap(categorical_correlations_sorted, 
            annot=True,
            cmap=cmap)
plt.title('Cramer\'s V Correlation with Match (Categorical Variables)')
plt.show()

In [ ]:
numeric_columns = cleaned_df.select_dtypes(include=['number']).columns
numeric_corr_match = cleaned_df[numeric_columns].corr()['match'].drop('match').sort_values(ascending=False)
corr_df = numeric_corr_match.to_frame('correlation_with_match')

cmap = sns.diverging_palette(110, 10, as_cmap=True)

n_variables = len(corr_df)
plt.figure(figsize=(8, max(6, n_variables * 0.4)))
sns.heatmap(corr_df, 
            annot=True,
            cmap=cmap)
plt.title('Numerical variables correlation with match')
plt.show()

In [ ]:
correlations_total = {}

for param in unified_d_parameters:
    correlations = {}
    correlations_total[param] = []
    for p in unified_d_parameters[param]:
        if p in cleaned_df.columns:
            correlations[p] = cramerV(cleaned_df[p], cleaned_df['match'])
            correlations_total[param].append({p: correlations[p]})
    
    if correlations:
        plot_df = pd.DataFrame(list(correlations.items()), 
                             columns=['Parameter', 'Correlation'])
        plot_df = plot_df.sort_values(by='Correlation', ascending=False)
        plot_df = plot_df.set_index('Parameter')
        
        plt.figure(figsize=(6, 2))
        sns.heatmap(plot_df, 
                    annot=True)
        plt.tight_layout()
        plt.show()

In [ ]:
for param in unified_parameters:
    correlations = {}
    for p in unified_parameters[param]:
        if p in cleaned_df.columns:
            correlations[p] = cleaned_df[p].corr(cleaned_df['match'])
            correlations_total[param].append({p: correlations[p]})
    
    if correlations:
        plot_df = pd.DataFrame(list(correlations.items()), 
                             columns=['Parameter', 'Correlation'])
        plot_df = plot_df.sort_values(by='Correlation', ascending=False)
        plot_df = plot_df.set_index('Parameter')
        
        plt.figure(figsize=(6, 2))
        sns.heatmap(plot_df, 
                    annot=True, 
                    center=0, 
                    fmt='.3f', 
                    cmap='RdBu_r',
                    cbar_kws={'label': 'Correlation'})
        plt.title(f'{param.title()} vs Match')
        plt.tight_layout()
        plt.show()

In [ ]:
correlations_total = {}

for param in correlations_total:
    flat_correlations = {}
    for item in correlations_total[param]:
        flat_correlations.update(item)
    plot_df = pd.DataFrame(list(flat_correlations.items()), 
                         columns=['Parameter', 'Correlation'])
    plot_df = plot_df.sort_values(by='Correlation', ascending=False)
    plot_df = plot_df.set_index('Parameter')
        
    plt.figure(figsize=(6, 4))
    sns.heatmap(plot_df, 
                    annot=True)
    plt.title(f'{param.title()} vs Match')
    plt.tight_layout()
    plt.show()

### Correlation between variables in the groups

Since we have the 8 variable for each parameter, half of which represented by d_, which probably means difference, so we consider to check their correlations too. We assume, that d_ and ordinar variables would have high correlation between them, so we can remove half of the variables, since it is not good to add highly correlated variables. That can lead to the overfitting or some other problems.

#### Encoding of categorical variables

All of our d_ variables are represented by string with ranges, that follows ascending order. Since there is no method, that would calculate the correlaition vetween the string-range categorical variable and numerical one, we consider to encode each ranges variable

In [ ]:
for param in unified_d_parameters.values():
  for p in param:
    unique_values = sorted(cleaned_df[p].unique())
    print(f"{p}: {unique_values}")
  print()

In [ ]:
encoder1 = OrdinalEncoder(categories=[['[0-15]', '[16-20]', '[21-100]']])
encoder2 = OrdinalEncoder(categories=[['[0-5]', '[6-8]', '[9-10]']])

for param in parameters:
  cleaned_df[f"d_{param}_important_o"] = encoder1.fit_transform(cleaned_df[[f"d_{param}_important_o"]])
  cleaned_df[f"d_{param}_important"] = encoder1.fit_transform(cleaned_df[[f"d_{param}_important"]])
  cleaned_df[f"d_p_eval_of_my_{param}"] = encoder2.fit_transform(cleaned_df[[f"d_p_eval_of_my_{param}"]])
  cleaned_df[f"d_my_eval_of_p_{param}"] = encoder2.fit_transform(cleaned_df[[f"d_my_eval_of_p_{param}"]])

In [ ]:
for param in unified_d_parameters.values():
  for p in param:
    unique_values = sorted(cleaned_df[p].unique())
    print(f"{p}: {unique_values}")
  print()

In [ ]:
for param in parameters:
  selected_group_columns = []
  selected_group_columns += unified_parameters[param]
  selected_group_columns += unified_d_parameters[param]

  corr_matrix = cleaned_df[selected_group_columns].corr()
  plt.figure(figsize=(8, 6))
  mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
  sns.heatmap(corr_matrix, annot=True, mask=mask, fmt='.2f', cmap='coolwarm', center=0)
  plt.title(f'Correlation Matrix for {param.title()} Parameters')
  plt.tight_layout()
  plt.show()

The theory about the d_ variables and the ordinary one was supported by correlation heatmaps. Each 2 pairs of veriable have high correlation, so we will drop all the variables, represented as d_ variables.The data are represented in more obscure format, which we cannot say with certainty what define.

In [ ]:
for param in unified_d_parameters.values():
  cleaned_df.drop(columns=param, inplace=True)

print(cleaned_df.columns.tolist())

### Clean columns

During analysis we also mentioned, that there are many hobbies and d_hobbies variables, which could easily be replaced with interests_correlate and d_interests_correlate. So we will drop them too

In [ ]:
interests_columns = ['sports', 'tvsports', 'exercise', 'dining', 'museums', 'art', 'hiking', 'gaming', 'clubbing', 'reading', 'tv', 'theater', 'movies', 'concerts', 'music', 'shopping', 'yoga', 'd_sports', 'd_tvsports', 'd_exercise', 'd_dining', 'd_museums', 'd_art', 'd_hiking', 'd_gaming', 'd_clubbing', 'd_reading', 'd_tv', 'd_theater', 'd_movies', 'd_concerts', 'd_music', 'd_shopping', 'd_yoga']
cleaned_df.drop(columns=interests_columns, inplace=True)

print(cleaned_df.columns.tolist())

In [ ]:
def plot_distributions(df):
  columns = df.select_dtypes(include=['number']).columns.tolist()

  col_num = 3
  row_num = math.ceil(len(columns) / col_num)

  _, axes = plt.subplots(nrows = row_num, ncols = col_num, figsize=(20, 4 * row_num))
  axes = axes.flatten()

  for i, col in enumerate(columns):
      sns.histplot(df[col], ax=axes[i], linewidth=0.3)
      axes[i].set_title(f'Distribution of {col}', fontsize=8)
      axes[i].set_xlabel(col, fontsize=8)
      axes[i].set_ylabel("Count", fontsize=8)

  plt.subplots_adjust(hspace=0.75, wspace=0.3)
  plt.tight_layout()
  plt.show()

In [ ]:
plot_distributions(cleaned_df)

In [ ]:
cleaned_df[cleaned_df['met'] == 3]['met'].count()

In [ ]:
cleaned_df['has_null'].groupby(cleaned_df['has_null']).agg("count")
cleaned_df['has_null'].unique()

We've spotted some issues with data:\
1. Met should be represented as binary variable, but somehow there appear 1 record with met == 3, which cannot be. Since it is only 1 record we will simply remove it

In [ ]:
cleaned_df = cleaned_df.drop(cleaned_df[cleaned_df['met'] == 3].index)
cleaned_df[cleaned_df['met'] == 3]['met'].count()

## Analyse numerical data

In [ ]:
numeric_columns = cleaned_df.select_dtypes(include=['number']).columns
correlation_matrix = cleaned_df[numeric_columns].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
numeric_columns = cleaned_df.select_dtypes(include=['number']).columns
corr_match = cleaned_df[numeric_columns].corr()['match'].drop('match').sort_values(ascending=False)

plt.figure(figsize=(6, 10))
sns.barplot(x=corr_match.values, y=corr_match.index)
plt.title('Correlation with match')
plt.tight_layout()
plt.show()

In [ ]:
numerical_columns = cleaned_df.select_dtypes(include=['number'])
groupby_match = numerical_columns.groupby("match").agg("count")
groupby_match

In [ ]:
if 'match' in numerical_columns.columns:
    plot_columns = numerical_columns.columns.drop('match')
else:
    plot_columns = numerical_columns.columns

n_cols = 3
n_rows = math.ceil(len(plot_columns) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(plot_columns):
    counts = cleaned_df.groupby('match')[col].apply(lambda x: x.notna().sum())

    bars = axes[i].bar(counts.index, counts.values, color=['red', 'blue'], alpha=0.7)
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_xlabel('Match')
    axes[i].set_ylabel('Count')
    axes[i].set_xticks([0, 1])

    for bar, count in zip(bars, counts.values):
        height = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                    f'{count}', ha='center', va='bottom', fontweight='bold')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Conclusional heatmap for each column

In [ ]:
cleaned_df.info()

## Corelation all available columns with match

we dont need has_null values columns

also all interest as movies or other we dont need becasue we have corelation interest column, but we will check them on heatmap

also all d_ columns we dont need because they are correlated with main columns and we can just dont use them(harder to process d columns for me)
(d_age a want to use because age is important for dating)(also from my udnerstanding we cant know d_ values before the date itself except for d_age)

also we dont want to use decision columns because they are will be get after date (we want to predict match before date)

also found a wave column, if we dont want to do a date we will not know wave

In [ ]:
column_to_drop = ['has_null', 'decision', 'decision_o', 'wave']
column_to_drop = column_to_drop + [col for col in cleaned_df.columns if col.startswith('d_')]
if 'd_age' in column_to_drop:
    column_to_drop.remove('d_age')
column_to_drop

In [ ]:
df_encoded_clean = data.copy().dropna().drop_duplicates()
for col in cleaned_df.select_dtypes(include=['object', 'category']).columns:
    df_encoded_clean[col] = cleaned_df[col].astype('category').cat.codes

df_encoded_clean = df_encoded_clean.drop(columns=column_to_drop)

corr = df_encoded_clean.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(15, 15))
sns.heatmap(corr, cmap="coolwarm")

## Check what columns have high corelation with match

In [ ]:
match_corr = corr['match'].abs().sort_values(ascending=False)
match_corr.drop('match', inplace=True)

In [ ]:
plt.figure(figsize=(8, 10))
sns.barplot(match_corr, orient='h')

## Correlation heatmap with top correlated columns

on this barplot we can see that most correlated columns with match, we will use them to check if they corelates beetween themselfs

In [ ]:
match_corr_top_columns = match_corr[match_corr > 0.1].index.tolist()
match_corr_top_columns

In [ ]:
corr = df_encoded_clean[match_corr_top_columns].corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(corr, cmap="coolwarm")

## Data preprocessing and normalization pipeline

I don't want to keep features that have high correlation with each other, because this can lead to model overfitting and other problems. Therefore, I will select only those features that have high correlation with the target variable (match), but low correlation with each other.

Also better to drop _o columns because they are self reported and can be biased

Also we decided that all feature selection will be done with pipeline

creating pipeline for preprocessing and feature selection for our best correlation columns with match

In [ ]:
def preprocessing_pipeline(cat_cols: list[str], num_cols: list[str]):
    return  compose.ColumnTransformer([
        ("num", pipeline.Pipeline([
            ("imp", impute.SimpleImputer(strategy="median")),
            ("sc", preprocessing.StandardScaler())
        ]), num_cols),
        ("cat", pipeline.Pipeline([
            ("imp", impute.SimpleImputer(strategy="most_frequent")),
            ("oh", preprocessing.OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ], remainder="drop")

In [ ]:
num_cols = data.select_dtypes(include="number").columns.tolist()
cat_cols = data.select_dtypes(exclude="number").columns.tolist()

num_cols = [col for col in num_cols if col in match_corr_top_columns]
cat_cols = [col for col in cat_cols if col in match_corr_top_columns]


In [ ]:
match_corr_top_columns

In [ ]:
num_cols

In [ ]:
cat_cols

# Data split

In [ ]:
x = data[num_cols + cat_cols].dropna()
y = data['match'].loc[x.index]

In [ ]:
x_train, x_test, y_train, y_test = model_selection.train_test_split(
    x, y, test_size=0.2, stratify=y
)

In [ ]:
x_train.shape[0], x_test.shape[0]

In [ ]:
y_test

# Train data preprocessing

In [ ]:
x_train_preprocessed = preprocessing_pipeline(cat_cols, num_cols).fit_transform(x_train)
x_test_preprocessed = preprocessing_pipeline(cat_cols, num_cols).fit_transform(x_test)

In [ ]:
x_train_preprocessed

In [ ]:
pca_all = decomposition.PCA(n_components=2)
x_train_pca_all = pca_all.fit_transform(x_train_preprocessed)
x_test_pca_all = pca_all.transform(x_test_preprocessed)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x=x_train_pca_all[:, 0], y=x_train_pca_all[:, 1], hue=y_train, 
                palette=['red', 'blue'], alpha=0.6, ax=axes[0])
axes[0].set_xlabel('First Principal Component')
axes[0].set_ylabel('Second Principal Component')
axes[0].set_title('PCA - Training Data (All Features)')
axes[0].legend(title='Match')

sns.scatterplot(x=x_test_pca_all[:, 0], y=x_test_pca_all[:, 1], hue=y_test, 
                palette=['red', 'blue'], alpha=0.6, ax=axes[1])
axes[1].set_xlabel('First Principal Component')
axes[1].set_ylabel('Second Principal Component')
axes[1].set_title('PCA - Test Data (All Features)')
axes[1].legend(title='Match')

# Wanddb configuration

In [ ]:
#os.environ['WANDB_API_KEY'] = 'use your key here'

In [ ]:
wandb_entity = "zneus_speeddating"
wandb_project = "zneus2025"
train_configs = [
    {
        "name": f"mlp_shallow_relu_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.003,
        "batch_size": 64,
        "epochs": 15,
        "hidden_dims": [128],
        "dropout": 0.05,
        "activations": ["RELU"]
    },
    {
        "name": f"mlp_compact_gelu_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.0015,
        "batch_size": 128,
        "epochs": 25,
        "hidden_dims": [256, 128],
        "dropout": 0.15,
        "activations": ["GELU"]
    },
    {
        "name": f"mlp_medium_mixed_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.0008,
        "batch_size": 256,
        "epochs": 30,
        "hidden_dims": [256, 128, 64],
        "dropout": 0.2,
        "activations": ["RELU", "GELU"]
    },
    {
        "name": f"mlp_deep_tanh_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.0004,
        "batch_size": 256,
        "epochs": 35,
        "hidden_dims": [512, 256, 128, 64],
        "dropout": 0.3,
        "activations": ["TANH"]
    },
    {
        "name": f"mlp_deep_v2_tanh_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.001,
        "batch_size": 512,
        "epochs": 50,
        "hidden_dims": [512, 256, 128, 64],
        "dropout": 0.3,
        "activations": ["TANH"]
    },
    {
        "name": f"mlp_deep_v3_tanh_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.01,
        "batch_size": 512,
        "epochs": 30,
        "hidden_dims": [512, 256, 128, 64, 32],
        "dropout": 0.2,
        "activations": ["TANH"]
    },
    {
        "name": f"mlp_narrow_fast_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.005,
        "batch_size": 32,
        "epochs": 20,
        "hidden_dims": [64, 32],
        "dropout": 0.1,
        "activations": ["RELU"]
    },
    {
        "name": f"mlp_wide_dropout_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.0006,
        "batch_size": 128,
        "epochs": 40,
        "hidden_dims": [512, 512, 256],
        "dropout": 0.25,
        "activations": ["GELU"]
    },
    {
        "name": f"mlp_balanced_combo_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.001,
        "batch_size": 96,
        "epochs": 25,
        "hidden_dims": [192, 96, 48],
        "dropout": 0.18,
        "activations": ["TANH", "RELU"]
    },
    {
        "name": f"mlp_large_batch_{time.strftime('%d_%m_%Y_%H%M')}",
        "learning_rate": 0.0009,
        "batch_size": 384,
        "epochs": 30,
        "hidden_dims": [384, 192, 96],
        "dropout": 0.22,
        "activations": ["RELU", "GELU"]
    }
]

## Train MLP models with Weights & Biases

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

train_features = torch.tensor(x_train_preprocessed, dtype=torch.float32)
test_features = torch.tensor(x_test_preprocessed, dtype=torch.float32)
train_targets = torch.tensor(y_train.to_numpy(), dtype=torch.float32).unsqueeze(1)
test_targets = torch.tensor(y_test.to_numpy(), dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(train_features, train_targets)
test_dataset = TensorDataset(test_features, test_targets)

# device = 'cuda'
device = 'cpu'

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for features, targets in dataloader:
        features, targets = features.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * targets.size(0)
    return total_loss / len(dataloader.dataset)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for features, targets in dataloader:
            features, targets = features.to(device), targets.to(device)
            logits = model(features)
            loss = criterion(logits, targets)
            total_loss += loss.item() * targets.size(0)
            preds = torch.sigmoid(logits)
            correct += ((preds >= 0.5) == (targets >= 0.5)).sum().item()
            total += targets.size(0)
    return total_loss / len(dataloader.dataset), correct / total

In [ ]:
#os.environ['WANDB_API_KEY'] = "api key"

In [ ]:
from model import MLP
from model import ActivationFunctions

for cfg in train_configs:
    run = wandb.init(
        project=wandb_project,
        config=cfg, 
        name=cfg["name"])
    config = wandb.config
    hidden_dims = tuple(config.hidden_dims)
    activation_layers = tuple(ActivationFunctions[name] for name in config.activations)
    model = MLP(
        input_dim=train_features.shape[1],
        hidden_dims=hidden_dims,
        output_dim=1,
        activations=activation_layers,
        dropout=config.dropout,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    criterion = torch.nn.BCEWithLogitsLoss()
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

    for epoch in range(config.epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        run.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        })
    run.finish()